# jiRAG — Jira Retrieval-Augmented Generation Assistant

This notebook serves as the primary entry point for the **jiRAG** project. It manages the end-to-end workflow, including:
*   **Dataset Generation & Validation**: Synthetic ticket creation and quality assurance.
*   **Preprocessing**: Text cleaning and chunking strategies.
*   **RAG & Vector Storage**: Document indexing and retrieval logic.
*   **Evaluation**: RAGAS and custom metrics for system performance.
*   **Model Fine-tuning**: QLoRA-based adaptation of LLMs.
*   **Agent & Jira Sync**: Tool-use for real-time synchronization with Jira.
*   **Deployment**: A final demo interface for support interaction.

## 0. Environment and Reproducibility

We begin by installing basic dependencies and setting up the software environment.

In [1]:
!pip install -q pandas numpy torch scikit-learn

In [2]:
import os
import json
import random
import hashlib
import platform
import sys
import csv
import re
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import sklearn

# --- 1. Global Configuration ---
CONFIG = {
    "project_name": "jiRAG",
    "project_version": "0.1.0",
    "random_seed": 42,
    "dataset_version": "v1",
    "prompt_version": "v1",
    "split_version": "v1",
    "index_version": "v1",
    "model_version": "base",
    "target_clean_tickets": 1000,
    "target_generated_tickets": 1100,
    "train_ratio": 0.70,
    "validation_ratio": 0.15,
    "test_ratio": 0.15,
    "default_chunking_strategy": "ticket",
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

SEED_INITIALIZED = False

### Reproducibility Settings

We define `set_global_seed` to ensure results are as consistent as possible across runs. Note that while these settings cover CPU and GPU operations, exact floating-point bitwise reproducibility on GPUs is not always guaranteed due to non-deterministic atomic operations in certain CUDA kernels.

In [3]:
def set_global_seed(seed):
    global SEED_INITIALIZED
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    SEED_INITIALIZED = True
    print(f"[INFO] Global seed set to: {seed}")

set_global_seed(CONFIG["random_seed"])

[INFO] Global seed set to: 42


### Directory Structure and Google Drive Mounting

We organize the project into a strict folder hierarchy stored on Google Drive to ensure persistence.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jiRAG")

# Define internal structure
PATHS = {
    "root": PROJECT_ROOT,
    "data_seeds": PROJECT_ROOT / "data" / "seeds",
    "data_raw": PROJECT_ROOT / "data" / "raw",
    "data_processed": PROJECT_ROOT / "data" / "processed",
    "data_splits": PROJECT_ROOT / "data" / "splits",
    "data_eval": PROJECT_ROOT / "data" / "evaluation",
    "data_manifests": PROJECT_ROOT / "data" / "manifests",
    "vector_store": PROJECT_ROOT / "vector_store",
    "models": PROJECT_ROOT / "models",
    "checkpoints": PROJECT_ROOT / "checkpoints",
    "reports_qa": PROJECT_ROOT / "reports" / "data_qa",
    "reports_eval": PROJECT_ROOT / "reports" / "evaluation",
    "reports_figs": PROJECT_ROOT / "reports" / "figures",
    "logs": PROJECT_ROOT / "logs",
    "exports": PROJECT_ROOT / "exports"
}

# Create directories safely
for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

print(f"[INFO] Project structure initialized at: {PROJECT_ROOT}")

Mounted at /content/drive
[INFO] Project structure initialized at: /content/drive/MyDrive/jiRAG


### Environment Inspection

Summary of the current runtime environment and project state.

In [5]:
def inspect_environment():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"
    print("="*50)
    print(f"Python Version:  {sys.version.split()[0]}")
    print(f"Pandas Version:  {pd.__version__}")
    print(f"NumPy Version:   {np.__version__}")
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available:  {torch.cuda.is_available()}")
    print(f"GPU Name:        {gpu_name}")
    print("-"*50)
    print(f"Project Root:    {PROJECT_ROOT}")
    print(f"Random Seed:     {CONFIG['random_seed']}")
    print(f"Dataset Version: {CONFIG['dataset_version']}")
    print("="*50)

inspect_environment()

Python Version:  3.13.15
Pandas Version:  2.2.3
NumPy Version:   2.1.3
PyTorch Version: 2.11.0+cpu
CUDA Available:  False
GPU Name:        N/A
--------------------------------------------------
Project Root:    /content/drive/MyDrive/jiRAG
Random Seed:     42
Dataset Version: v1


### Config Management

Saving the configuration to the manifests folder and checking for existing versions to prevent accidental drift.

In [6]:
config_file = PATHS["data_manifests"] / f"project_config_{CONFIG['dataset_version']}.json"

def sync_config(config, file_path):
    if file_path.exists():
        with open(file_path, 'r', encoding='utf-8') as f:
            existing_config = json.load(f)

        # Compare keys excluding timestamp
        diff = {k: config[k] for k in config if k != 'created_at_utc' and config.get(k) != existing_config.get(k)}

        if not diff:
            print("[INFO] Configuration matches existing file.")
        else:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            backup_path = file_path.with_suffix(f".bak_{timestamp}.json")
            file_path.rename(backup_path)
            print(f"[WARN] Config mismatch found. Backup created at {backup_path.name}. Updating file...")
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(config, f, indent=2)
    else:
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(config, f, indent=2)
        print(f"[INFO] Configuration saved to {file_path.name}")

sync_config(CONFIG, config_file)

[INFO] Configuration matches existing file.


### Utility Functions

Helper functions for data integrity and experiment tracking.

In [7]:
def calculate_sha256(file_path):
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

def save_run_metadata(stage_name, extra_metadata=None):
    metadata = {
        "stage_name": stage_name,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "project_version": CONFIG["project_version"],
        "dataset_version": CONFIG["dataset_version"],
        "prompt_version": CONFIG["prompt_version"],
        "split_version": CONFIG["split_version"],
        "index_version": CONFIG["index_version"],
        "model_version": CONFIG["model_version"],
        "random_seed": CONFIG["random_seed"],
        "python_version": platform.python_version(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "sklearn_version": sklearn.__version__,
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "extra": extra_metadata or {}
    }

    log_filename = f"run_{stage_name}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}.json"
    log_path = PATHS["logs"] / log_filename

    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2)
    print(f"[INFO] Metadata logged to {log_filename}")

save_run_metadata("project_bootstrap")

[INFO] Metadata logged to run_project_bootstrap_20260825_154732.json


### Bootstrap Validation

In [8]:
def validate_bootstrap():
    errors = []
    if not SEED_INITIALIZED:
        errors.append("SEED_INITIALIZED is False. Call set_global_seed() first.")

    for name, p in PATHS.items():
        if not p.exists():
            errors.append(f"Directory missing: {name} at {p}")

    total_ratio = CONFIG["train_ratio"] + CONFIG["validation_ratio"] + CONFIG["test_ratio"]
    if abs(total_ratio - 1.0) > 1e-9:
        errors.append(f"Split ratios do not equal 1.0 (Sum: {total_ratio})")

    if CONFIG["target_generated_tickets"] <= CONFIG["target_clean_tickets"]:
        errors.append("target_generated_tickets must be > target_clean_tickets")

    if not config_file.exists():
        errors.append("Project config file was not created.")

    if errors:
        print("❌ Bootstrap Validation Failed:")
        for err in errors:
            print(f"  - {err}")
    else:
        print("✅ jiRAG project bootstrap completed successfully")

validate_bootstrap()

✅ jiRAG project bootstrap completed successfully


## 1. Seed Dataset Loading and Validation

In [9]:
import csv
import re
import json
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import numpy as np

SEED_FILE_PATH = PATHS["data_seeds"] / "dataset_pilot_v2.csv"
OUTPUT_CSV = PATHS["data_processed"] / "seed_tickets_normalized_v1.csv"
MANIFEST_PATH = PATHS["data_manifests"] / "seed_dataset_manifest_v1.json"

REQUIRED_DESC_HEADINGS = [
    "CONTEXT", "ISSUE OR REQUEST", "EXPECTED BEHAVIOR",
    "ACTUAL BEHAVIOR", "INVESTIGATION AND FINDINGS",
    "RESOLUTION", "VALIDATION"
]

# 0. Clean up previous failed outputs if they exist
if OUTPUT_CSV.exists():
    OUTPUT_CSV.unlink()

# 1. Verify file existence
if not SEED_FILE_PATH.exists():
    raise FileNotFoundError(f"Seed file missing at {SEED_FILE_PATH}")

# 2. Structure Validation (csv module)
with open(SEED_FILE_PATH, mode='r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = list(reader)

orig_col_count = len(header)
orig_row_count = len(rows)
labels_count = header.count("Labels")
expected_others = ["Summary", "Description", "Work Type", "Priority", "Status", "Component"]

struct_errors = []
if orig_col_count != 9: struct_errors.append(f"Expected 9 columns, found {orig_col_count}")
if labels_count != 3: struct_errors.append(f"Expected 3 'Labels' headers, found {labels_count}")
if orig_row_count != 30: struct_errors.append(f"Expected 30 data rows, found {orig_row_count}")

# 3. Load and Normalize
raw_df = pd.read_csv(SEED_FILE_PATH)
rename_map = {'Labels': 'ticket_id', 'Labels.1': 'incident_family', 'Labels.2': 'solution_status'}
seed_df = raw_df.rename(columns=rename_map)

validation_failures = []
pii_findings = []

# 4-6. Enhanced Validation
for idx, row in seed_df.iterrows():
    row_errs = []
    t_id = str(row['ticket_id'])

    # Empty check (NaN or whitespace only)
    for col in seed_df.columns:
        val = row[col]
        if pd.isna(val) or (isinstance(val, str) and not val.strip()):
            row_errs.append(f"Field '{col}' is empty")

    # ID/Summary logic
    if not re.match(r"^tckt-\d+$", t_id): row_errs.append("Invalid ticket_id format")
    if seed_df['ticket_id'].duplicated().iloc[idx]: row_errs.append("Duplicate ticket_id")
    if seed_df['Summary'].duplicated().iloc[idx]: row_errs.append("Duplicate Summary")

    # Precise Description Heading Validation
    desc = str(row['Description'])
    words = desc.split()
    if not (90 <= len(words) <= 220): row_errs.append(f"Word count {len(words)} outside range")

    # Find headings as standalone lines
    found_indices = []
    for h in REQUIRED_DESC_HEADINGS:
        pattern = rf"^{re.escape(h)}\s*$"
        matches = list(re.finditer(pattern, desc, re.MULTILINE))
        if len(matches) != 1:
            row_errs.append(f"Heading '{h}' must appear exactly once as standalone line")
        else:
            found_indices.append(matches[0].start())

    if len(found_indices) == len(REQUIRED_DESC_HEADINGS):
        if found_indices != sorted(found_indices):
            row_errs.append("Description headings are not in the required order")

    # Categorical/Workflow Logic
    # Jira Status represents workflow completion, while solution_status represents quality.
    # A ticket may be Done with a documented workaround or partial solution.
    sol, stat = row['solution_status'], row['Status']
    if row['Work Type'] not in ['Bug', 'Task', 'Story']: row_errs.append("Invalid Work Type")
    if sol == 'solution-verified' and stat != 'Done': row_errs.append("solution-verified must use Done")
    if sol == 'solution-unresolved' and stat == 'Done': row_errs.append("solution-unresolved cannot be Done")

    # PII Precise Scan
    content = f"{row['Summary']} {desc}"
    if re.search(r"[\w\.-]+@[\w\.-]+\.\w+", content): pii_findings.append(f"{t_id}: Email")
    # Credential scan requiring assignment
    if re.search(r"(?i)(password|passwd|secret|key|token)[\s:=]+[^\s\n]{4,}", content):
        pii_findings.append(f"{t_id}: Exposed credential")

    if row_errs: validation_failures.append({"Row": idx, "Ticket": t_id, "Errors": row_errs})

# 7. Reporting
seed_dataset_valid = (len(validation_failures) == 0 and len(struct_errors) == 0)
print(f"Dataset Dimensions: {seed_df.shape}")
display(seed_df.drop(columns=['Description']).head(3))
print(f"\nFamilies:\n{seed_df['incident_family'].value_counts()}")
print(f"\nSolution Status:\n{seed_df['solution_status'].value_counts()}")
print(f"\nWord Counts:\n{seed_df['Description'].apply(lambda x: len(str(x).split())).describe()}")

if pii_findings: print(f"\n[PII SCAN] Findings: {pii_findings}")
if not seed_dataset_valid:
    print(f"\n❌ Validation Failures Found: {struct_errors}")
    display(pd.DataFrame(validation_failures))

# 8-11. Manifest and Persistence
seed_sha256 = calculate_sha256(SEED_FILE_PATH)
manifest = {
    "dataset_role": "reviewed_seed_dataset",
    "dataset_version": CONFIG["dataset_version"],
    "original_file_name": SEED_FILE_PATH.name,
    "original_file_path": str(SEED_FILE_PATH),
    "sha256": seed_sha256,
    "row_count": orig_row_count, "column_count": orig_col_count,
    "incident_family_count": seed_df['incident_family'].nunique(),
    "validation_passed": seed_dataset_valid,
    "validation_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "random_seed": CONFIG["random_seed"],
    "normalized_schema": list(seed_df.columns),
    "allowed_values": {"Status": ["To Do", "In Progress", "Done"], "solution_status": ["solution-verified", "solution-partial", "solution-workaround", "solution-unresolved"]},
    "required_description_headings": REQUIRED_DESC_HEADINGS,
    "validation_rule_version": "1.1"
}

with open(MANIFEST_PATH, 'w') as f: json.dump(manifest, f, indent=2)

if seed_dataset_valid:
    seed_df.to_csv(OUTPUT_CSV, index=False)
    save_run_metadata("seed_dataset_validation", {"path": str(OUTPUT_CSV), "sha256": seed_sha256, "rows": len(seed_df), "result": "passed"})
    if MANIFEST_PATH.exists() and OUTPUT_CSV.exists():
        print("\n✅ Seed dataset validated successfully: 30 tickets are ready for generation")

Dataset Dimensions: (30, 9)


,Summary,Work Type,Priority,Status,Component,ticket_id,incident_family,solution_status
0,Opportunity approval flow creates duplicate re...,Bug,High,Done,Opportunity Automation,tckt-0005,family-salesforce-flow,solution-verified
1,Case escalation flow repeatedly updates its ow...,Bug,Highest,Done,Case Escalation Flow,tckt-0006,family-salesforce-flow,solution-verified
2,Scheduled renewal actions intermittently remai...,Task,Medium,In Progress,Renewal Scheduling,tckt-0007,family-salesforce-flow,solution-unresolved



Families:
incident_family
family-salesforce-flow          3
family-crm-integration          3
family-salesforce-security      3
family-soql-analytics           3
family-email-activity           3
family-crm-record-management    3
family-revenue-operations       3
family-api-platform             3
family-ai-to-soql               3
family-data-migration           3
Name: count, dtype: int64

Solution Status:
solution_status
solution-verified      17
solution-unresolved     5
solution-partial        5
solution-workaround     3
Name: count, dtype: int64

Word Counts:
count     30.000000
mean     142.633333
std       14.045452
min      115.000000
25%      137.000000
50%      141.000000
75%      147.750000
max      191.000000
Name: Description, dtype: float64

[PII SCAN] Findings: ['tckt-0026: Exposed credential']
[INFO] Metadata logged to run_seed_dataset_validation_20260825_154734.json

✅ Seed dataset validated successfully: 30 tickets are ready for generation
